In [6]:
def extract_player_info(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find('table', {'id': 'team_injuries'})
    rows = table.find_all('tr')
    player_injuries = []
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 7:
            player_name = cols[0].get_text().strip()
            player_injury = cols[2].get_text().strip()
            player_status = cols[4].get_text().strip()
            player_injuries.append({
                "player_name": player_name,
                "player_injury": player_injury,
                "player_status": player_status
            })
    return player_injuries


In [7]:
player_schema = StructType([
    StructField("player_name", StringType(), True),
    StructField("player_injury", StringType(), True),
    StructField("player_status", StringType(), True),
])


In [8]:
# URLs for player injury
injury_url = "https://www.pro-football-reference.com/teams/nwe/2014_injuries.htm"

# Fetch HTML content
injury_html = fetch_html_content(injury_url)

# Parse HTML content and extract data
injury_data = extract_player_info(injury_html)

# Create DataFrame from the extracted data
injury_df = spark.createDataFrame(injury_data, schema=player_schema)

# Perform necessary processing or analysis using PySpark DataFrame APIs
# ...

# Save the DataFrame to a CSV file
injury_df.write.csv("player_injury_data.csv", header=True, mode="overwrite")

# Stop the Spark session
spark.stop()


AttributeError: 'NoneType' object has no attribute 'sc'

In [9]:
import requests
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# Function to fetch HTML content using requests
def fetch_html_content(url):
    response = requests.get(url)
    return response.content

# Function to parse the HTML content using BeautifulSoup and extract player injury information
def extract_player_info(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find('table', {'id': 'team_injuries'})
    rows = table.find_all('tr')
    player_injuries = []
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 7:
            player_name = cols[0].get_text().strip()
            player_injury = cols[2].get_text().strip()
            player_status = cols[4].get_text().strip()
            player_injuries.append({
                "player_name": player_name,
                "player_injury": player_injury,
                "player_status": player_status
            })
    return player_injuries

# Initialize Spark session
spark = SparkSession.builder.appName("NFLPlayerData").getOrCreate()

# URL for player injury
injury_url = "https://www.pro-football-reference.com/teams/nwe/2014_injuries.htm"

# Fetch HTML content
injury_html = fetch_html_content(injury_url)

# Parse HTML content and extract data
injury_data = extract_player_info(injury_html)

# Define the schema for the DataFrame
player_schema = StructType([
    StructField("player_name", StringType(), True),
    StructField("player_injury", StringType(), True),
    StructField("player_status", StringType(), True),
])

# Create DataFrame from the extracted data
injury_df = spark.createDataFrame(injury_data, schema=player_schema)

# Display the DataFrame
injury_df.show()

# Save the DataFrame to a CSV file
injury_df.write.csv("player_injury_data.csv", header=True, mode="overwrite")

# Stop the Spark session
spark.stop()


+-----------+-------------+-------------+
|player_name|player_injury|player_status|
+-----------+-------------+-------------+
|           |             |             |
|           |             |             |
|           |             |             |
|           |             |             |
|          P|             |             |
|          S|            S|            P|
|          Q|            Q|             |
|           |             |             |
|           |             |             |
|           |            Q|            Q|
|           |            Q|             |
|           |            Q|            Q|
|           |             |             |
|           |             |            Q|
|           |             |            Q|
|           |            P|             |
|           |             |            Q|
|         IR|           IR|           IR|
|         IR|           IR|           IR|
|           |             |             |
+-----------+-------------+-------

# NBA DATASET FROM KAAGLE
available at https://www.kaggle.com/code/jaseziv83/extensive-nba-injuries-deep-dive-eda

In [5]:
from pyspark.sql import SparkSession

# Set the Google Cloud service account JSON key file path in the Hadoop configuration
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", "/home/hduser/Downloads/gcs.json")

# Add the necessary dependencies for GCS file system access
spark.conf.set("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.1")

# Create a SparkSession
spark = SparkSession.builder.appName('GCSFilesRead').getOrCreate()

# Continue with the rest of your code...
bucket_name = "sba22246ca2"
path = f"gs://{bucket_name}/data/injuries_2010-2020.csv"

df = spark.read.csv(path, header=True)
df.show()

+----------+------------+--------+----------------+--------------------+
|      Date|        Team|Acquired|    Relinquished|               Notes|
+----------+------------+--------+----------------+--------------------+
|2010-10-03|       Bulls|    null|   Carlos Boozer|fractured bone in...|
|2010-10-06|     Pistons|    null|   Jonas Jerebko|torn right Achill...|
|2010-10-06|     Pistons|    null|   Terrico White|broken fifth meta...|
|2010-10-08|     Blazers|    null|      Jeff Ayres|torn ACL in right...|
|2010-10-08|        Nets|    null|     Troy Murphy|strained lower ba...|
|2010-10-08|     Pistons|    null|   Jonas Jerebko|surgery to repair...|
|2010-10-08|     Pistons|    null|   Terrico White|surgery on right ...|
|2010-10-09|     Nuggets|    null|   Al Harrington|partially torn pl...|
|2010-10-12|       Bucks|    null|Darington Hobson|surgery on left h...|
|2010-10-12|       Kings|    null|Samuel Dalembert|strained left gro...|
|2010-10-17|     Bobcats|    null| Dominic McGuire|

In [6]:
# drop rows that only contain acl or knee
from pyspark.sql import functions as F

# Filter the DataFrame to include only rows where the Notes column contains "knee" or "acl"
df = df.filter(F.lower(df.Notes).like("%knee%") | F.lower(df.Notes).like("%acl%"))

# Show the DataFrame
df.show()


+----------+---------+--------+----------------+--------------------+
|      Date|     Team|Acquired|    Relinquished|               Notes|
+----------+---------+--------+----------------+--------------------+
|2010-10-08|  Blazers|    null|      Jeff Ayres|torn ACL in right...|
|2010-10-25|  Thunder|    null|   Nick Collison|left knee injury ...|
|2010-10-26|  Blazers|    null|       Greg Oden|placed on IL with...|
|2010-10-26|  Blazers|    null|  Joel Przybilla|placed on IL plac...|
|2010-10-26|  Celtics|    null|Kendrick Perkins|placed on IL reco...|
|2010-10-26|   Lakers|    null|    Andrew Bynum|placed on IL reco...|
|2010-10-27|    Bucks|    null|    Michael Redd|placed on IL reco...|
|2010-10-27|   Knicks|    null|Kelenna Azubuike|placed on IL reco...|
|2010-10-27|  Nuggets|    null|  Chris Andersen|placed on IL reco...|
|2010-10-27|  Nuggets|    null|   Kenyon Martin|placed on IL reco...|
|2010-10-28|    Hawks|    null|   Maurice Evans|right knee inflam...|
|2010-10-28|  Wizard

In [7]:
num_rows = df.count()
print(f'There are {num_rows} rows in the DataFrame.')


There are 2730 rows in the DataFrame.


In [9]:
num_acquired = df.filter(df.Acquired.isNotNull()).count()
print(f'There are {num_acquired} non-null values in the Acquired column.')


There are 2 non-null values in the Acquired column.


In [12]:
df.filter(df.Acquired.isNotNull()).show()


+----------+-------+-----------------+------------+--------------------+
|      Date|   Team|         Acquired|Relinquished|               Notes|
+----------+-------+-----------------+------------+--------------------+
|2013-04-27|Thunder|Russell Westbrook|        null|surgery on right ...|
|2018-10-07|  Spurs|  Dejounte Murray|        null|torn ACL in right...|
+----------+-------+-----------------+------------+--------------------+



In [13]:
df = df.drop("Acquired")


In [11]:
from pyspark.sql.functions import min, max

# Compute the minimum and maximum values for the 'created_at' column
min_date = df.agg(min('date')).collect()[0][0]
max_date = df.agg(max('date')).collect()[0][0]

print("Earliest Date:", min_date)
print("Latest Date:", max_date)

Earliest Date: 2010-10-08
Latest Date: 2020-09-23


In [20]:
import requests
from bs4 import BeautifulSoup

def fetch_html_content(url):
    try:
        response = requests.get(url)
        return response.content
    except requests.exceptions.RequestException as e:
        print(e)
        return None

def fetch_player_info(player_name):
    # URL for the Wikipedia page of the player
    player_url = f"https://en.wikipedia.org/wiki/{player_name.replace(' ', '_')}"

    # Get the HTML content of the page
    player_html_content = fetch_html_content(player_url)

    # Parse the HTML with BeautifulSoup
    soup = BeautifulSoup(player_html_content, 'html.parser')

    # Find the infobox
    infobox = soup.find('table', {'class': 'infobox'})

    # Extract the desired info if it exists
    if infobox:
        bday = infobox.find('span', {'class': 'bday'}).text if infobox.find('span', {'class': 'bday'}) else None
        weight = infobox.find('th', text='Weight').find_next_sibling('td').text if infobox.find('th', text='Weight') else None
        height = infobox.find('th', text='Height').find_next_sibling('td').text if infobox.find('th', text='Height') else None
        listed_weight = infobox.find('th', text='Listed weight').find_next_sibling('td').text if infobox.find('th', text='Listed weight') else None
        listed_height = infobox.find('th', text='Listed height').find_next_sibling('td').text if infobox.find('th', text='Listed height') else None
    else:
        bday, weight, height, listed_weight, listed_height = None, None, None, None, None

    return {
        'player_name': player_name,
        'birthday': bday,
        'weight': weight if weight is not None else listed_weight,
        'height': height if height is not None else listed_height
    }


players = df.select('Relinquished').rdd.flatMap(lambda x: x).collect()
# Filter out None values
players = [player for player in players if player is not None]

players_info_list = []

for player in players:  
    player_info = fetch_player_info(player)
    players_info_list.append(player_info)



In [22]:
for player_info in players_info_list[:5]:  # Only print the first 5 players' info
    print(player_info)


{'player_name': 'Jeff Ayres', 'birthday': '1987-04-29', 'weight': '240\xa0lb (109\xa0kg)', 'height': '6\xa0ft 9\xa0in (2.06\xa0m)'}
{'player_name': 'Nick Collison', 'birthday': '1980-10-26', 'weight': '255\xa0lb (116\xa0kg)', 'height': '6\xa0ft 10\xa0in (2.08\xa0m)'}
{'player_name': 'Greg Oden', 'birthday': '1988-01-22', 'weight': '273\xa0lb (124\xa0kg)', 'height': '7\xa0ft 0\xa0in (2.13\xa0m)'}
{'player_name': 'Joel Przybilla', 'birthday': '1979-10-10', 'weight': '245\xa0lb (111\xa0kg)', 'height': '7\xa0ft 1\xa0in (2.16\xa0m)'}
{'player_name': 'Kendrick Perkins', 'birthday': '1984-11-10', 'weight': '270\xa0lb (122\xa0kg)', 'height': '6\xa0ft 10\xa0in (2.08\xa0m)'}


In [21]:
df.show()

+----------+---------+----------------+--------------------+
|      Date|     Team|    Relinquished|               Notes|
+----------+---------+----------------+--------------------+
|2010-10-08|  Blazers|      Jeff Ayres|torn ACL in right...|
|2010-10-25|  Thunder|   Nick Collison|left knee injury ...|
|2010-10-26|  Blazers|       Greg Oden|placed on IL with...|
|2010-10-26|  Blazers|  Joel Przybilla|placed on IL plac...|
|2010-10-26|  Celtics|Kendrick Perkins|placed on IL reco...|
|2010-10-26|   Lakers|    Andrew Bynum|placed on IL reco...|
|2010-10-27|    Bucks|    Michael Redd|placed on IL reco...|
|2010-10-27|   Knicks|Kelenna Azubuike|placed on IL reco...|
|2010-10-27|  Nuggets|  Chris Andersen|placed on IL reco...|
|2010-10-27|  Nuggets|   Kenyon Martin|placed on IL reco...|
|2010-10-28|    Hawks|   Maurice Evans|right knee inflam...|
|2010-10-28|  Wizards|     Josh Howard|placed on IL with...|
|2010-10-29|  Celtics| Jermaine O'Neal|placed on IL with...|
|2010-10-29|    Hawks|  

In [23]:
player_info

{'player_name': 'Kendrick Perkins',
 'birthday': '1984-11-10',
 'weight': '270\xa0lb (122\xa0kg)',
 'height': '6\xa0ft 10\xa0in (2.08\xa0m)'}

Listed height	6 ft 10 in (2.08 m)
Listed weight	270 lb (122 kg)